# Colab runner: train / evaluate on a free GPU  (issue #10)

Same idea as `kaggle_runner.ipynb`, with Google Drive for persistence: **checkpoints are written to Drive**, so
`--resume` continues a run in a new session after Colab disconnects.

**Open it:** colab.research.google.com → **File → Open notebook → GitHub** →
`ErenKerimoglu/CS464_Project` → `notebooks/colab_runner.ipynb`. Then **Runtime → Change runtime type → T4 GPU**.

**Data (pick one in cell 1, `DATA_SOURCE`)**
* `"drive"`: the processed cache is in `DRIVE_DIR/processed/` (`images_256.npy`, `meta.csv`). If a teammate shared a
  folder with you, open it in Drive → **Organize → Add shortcut** → *My Drive*, and name the shortcut `CS464_galaxy`.
  It is copied to the Colab disk once per session (≈ 1–2 min), because reading it from Drive directly is slow.
* `"kaggle"`: downloads the team Kaggle dataset with your Kaggle API token. kaggle.com → Settings → API →
  *Create New Token* gives a token `KGAT_...`. Add it as a Colab secret (🔑 icon on the left) named
  **`KAGGLE_API_TOKEN`** and enable notebook access. Old-style `KAGGLE_USERNAME` + `KAGGLE_KEY` secrets also work.
  **Never paste the token into a cell, an issue or a chat.** With `SAVE_TO_DRIVE = True` the download is also stored
  in `DRIVE_DIR/processed/`.
* `"auto"` (default): `"drive"` if `DRIVE_DIR/processed/images_256.npy` exists, otherwise `"kaggle"`.

If Colab says no GPU backend is available, try again later, or queue the run on Kaggle (*Save Version*).

**Every run:** add a row to `docs/RUNS.md`, edit `RUN_CMDS` (cell 4), **Runtime → Run all**. After a disconnect:
add `--resume` to the same commands and run all again. Results are exported to `DRIVE_DIR/results_export.zip`.

## 1 · Parameters, Drive, clone

In [ ]:
BRANCH = "main"                                   # branch, tag or commit hash to run
DRIVE_DIR = "/content/drive/MyDrive/CS464_galaxy" # checkpoints/, processed/ and results_export/ live here
DATA_SOURCE = "auto"                              # "auto", "drive" or "kaggle" (see the top of the notebook)
KAGGLE_DATASET = "<owner>/galaxy10-decals-processed-v1"   # owner = the Kaggle username of the dataset's creator
SAVE_TO_DRIVE = True                              # "kaggle" source: also keep a copy in DRIVE_DIR/processed

REPO_URL = "https://github.com/ErenKerimoglu/CS464_Project.git"
REPO_DIR = "/content/CS464_Project"

import os, subprocess, sys, time
SESSION_START = time.time()
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"], check=False)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("running commit", COMMIT)

## 2 · Install the package (torch is preinstalled on Colab)

In [ ]:
# "all" (incl. escnn) fails on Kaggle's Python 3.13. Add ",equivariant" only for escnn runs on Python <= 3.12 (#10).
EXTRAS = "deep,classical,tuning"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)  # stops on failure
sys.path.insert(0, REPO_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3 · Data and paths (checkpoints on Drive)

In [ ]:
import shutil
LOCAL_PROCESSED = "/content/data/processed"
if DATA_SOURCE == "auto":
    DATA_SOURCE = "drive" if os.path.exists(f"{DRIVE_DIR}/processed/images_256.npy") else "kaggle"
    print("data source:", DATA_SOURCE)
if DATA_SOURCE == "kaggle" and not os.path.exists(f"{LOCAL_PROCESSED}/images_256.npy"):
    if "<owner>" in KAGGLE_DATASET:
        raise ValueError("set KAGGLE_DATASET in cell 1 to '<owner's Kaggle username>/galaxy10-decals-processed-v1'")
    from google.colab import userdata

    def _secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None

    if _secret("KAGGLE_API_TOKEN"):                       # new-style token (KGAT_...)
        os.environ["KAGGLE_API_TOKEN"] = _secret("KAGGLE_API_TOKEN")
    elif _secret("KAGGLE_USERNAME") and _secret("KAGGLE_KEY"):   # old-style kaggle.json
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
    else:
        raise RuntimeError("add the Colab secret KAGGLE_API_TOKEN (🔑 icon) and enable notebook access")
    !pip install -q -U kaggle
    !kaggle datasets download -d {KAGGLE_DATASET} -p /content/data --unzip
    if SAVE_TO_DRIVE:
        os.makedirs(f"{DRIVE_DIR}/processed", exist_ok=True)
        for name in ("images_256.npy", "meta.csv"):
            shutil.copyfile(f"{LOCAL_PROCESSED}/{name}", f"{DRIVE_DIR}/processed/{name}")
        print("saved a copy to", f"{DRIVE_DIR}/processed")

from galaxy.utils.env import setup_env
paths = setup_env(drive_dir=DRIVE_DIR)   # copies DRIVE_DIR/processed to /content/data/processed if needed
if not os.path.exists(os.path.join(paths["processed_dir"], "images_256.npy")):
    raise FileNotFoundError(f"images_256.npy not found: put it in {DRIVE_DIR}/processed or use DATA_SOURCE='kaggle'")

## 4 · Commands to run

One or more commands, run in order from the repo root. A failing command stops the rest.
Check `docs/RUNS.md` first so nobody trains the same run twice. After a disconnect, add `--resume`.

In [ ]:
RUN_CMDS = [
    "python scripts/train.py --config configs/models/small_cnn.yaml --seed 0 --set train.max_epochs=1 data.fraction=0.1 data.num_workers=2",
    # "python scripts/train.py --config configs/models/resnet50.yaml --seed 0 --set experiment=prelim run_name=resnet50_rotflip data.num_workers=2",
    # "python scripts/train.py --config configs/models/resnet50.yaml --seed 0 --set data.num_workers=2 --resume",
]

## 5 · Run

In [ ]:
from galaxy.utils.env import run_commands
run_commands(RUN_CMDS, cwd=REPO_DIR)

## 6 · Export results → `DRIVE_DIR/results_export.zip` and print the git steps

In [ ]:
from galaxy.utils.env import export_results, results_instructions
EXPORT_DIR = f"{DRIVE_DIR}/results_export"
run_dirs = export_results(REPO_DIR, EXPORT_DIR, newer_than=SESSION_START)
print(results_instructions(run_dirs, EXPORT_DIR + ".zip", COMMIT))